# Data Cleaning and Quality Analysis

**Oasis Infobyte Data Analytics Internship — Level 1, Task 3**

This notebook demonstrates a complete data-cleaning workflow using the Titanic passenger dataset.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the Dataset

The Titanic dataset is one of the datasets suggested in the internship task sheet for data-cleaning practice.


In [ ]:
df = sns.load_dataset("titanic")

print("Dataset shape:", df.shape)
display(df.head())


## 3. Initial Data Quality Report


In [ ]:
def data_quality_report(data):
    report = pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "null_count": data.isna().sum(),
        "null_percent": (data.isna().mean() * 100).round(2),
        "unique_values": data.nunique(dropna=False)
    })
    return report

quality_before = data_quality_report(df)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())

display(quality_before)


## 4. Numeric Range and Categorical Checks

Numeric ranges help identify values that may be impossible or unusually extreme. Categorical values are inspected for inconsistent formatting before standardization.


In [ ]:
print("Numeric summary:")
display(df.describe().T)

for column in df.select_dtypes(include=["object", "category"]).columns:
    print(f"\n{column} unique values:")
    print(df[column].dropna().astype(str).unique())


## 5. Define Cleaning Decisions

The cleaning decisions are documented before modifications are applied.

**Missing values**
- Age: median imputation.
- Fare: median imputation.
- Embarked: mode imputation.
- Deck/Cabin information: represented through a `CabinKnown` indicator.

**Duplicates**
- Exact duplicate rows are removed.

**Categorical standardization**
- Text is stripped of surrounding whitespace.
- Sex is standardized to `Male` and `Female`.
- Embarked codes are normalized to uppercase.

**Outliers**
- The IQR method is used.
- Selected numeric outliers are capped at the IQR bounds instead of deleting observations, preserving the dataset's row-level information.


## 6. Remove Exact Duplicates


In [ ]:
clean = df.copy()

duplicates_before = clean.duplicated().sum()
clean = clean.drop_duplicates().reset_index(drop=True)
duplicates_removed = duplicates_before - clean.duplicated().sum()

print("Duplicate rows before:", duplicates_before)
print("Duplicate rows removed:", duplicates_removed)


## 7. Handle Missing Values


In [ ]:
# Create a useful cabin availability indicator before dropping the high-missing Cabin field.
clean["CabinKnown"] = clean["deck"].notna().astype(int)

# Numeric columns: median imputation
for column in ["age", "fare"]:
    clean[column] = clean[column].fillna(clean[column].median())

# Categorical column: mode imputation
clean["embarked"] = clean["embarked"].fillna(clean["embarked"].mode()[0])

print("Missing values after imputation:")
display(clean.isna().sum().to_frame("missing_values"))


### Why these choices?

Median imputation is less sensitive to extreme values than the mean. Mode imputation is appropriate for a categorical field such as embarkation port.

The original `deck` field has substantial missingness. Rather than inventing deck labels, the workflow extracts the useful information of whether cabin/deck information exists and stores it in `CabinKnown`.


## 8. Standardize Categorical Formatting


In [ ]:
clean["sex"] = (
    clean["sex"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"male": "Male", "female": "Female"})
)

clean["embarked"] = (
    clean["embarked"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Convert selected categorical columns to category dtype.
for column in ["sex", "embarked", "class", "who", "alone"]:
    if column in clean.columns:
        clean[column] = clean[column].astype("category")

display(clean[["sex", "embarked"]].head())


## 9. Data Type Correction


In [ ]:
numeric_columns = ["age", "fare", "sibsp", "parch", "pclass", "survived"]

for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

# Restore nullable integer types for count/class columns.
for column in ["sibsp", "parch", "pclass", "survived"]:
    clean[column] = clean[column].astype("Int64")

print("Data types after correction:")
display(clean.dtypes.to_frame("dtype"))


## 10. Detect Outliers Using the IQR Method

For a numeric variable:

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Values outside these bounds are flagged as potential outliers.


In [ ]:
outlier_columns = ["age", "fare", "sibsp", "parch"]

outlier_report = []

for column in outlier_columns:
    series = clean[column].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((series < lower) | (series > upper)).sum()

    outlier_report.append({
        "Column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Potential Outliers": int(count)
    })

outlier_report = pd.DataFrame(outlier_report).round(2)
display(outlier_report)


## 11. Treat Outliers

Potential outliers are capped at the IQR bounds for the selected numeric variables.

This avoids deleting passengers simply because their values are extreme. The decision is documented and can be changed to removal if a downstream use case requires it.


In [ ]:
outlier_counts_before = {}

for column in outlier_columns:
    q1 = clean[column].quantile(0.25)
    q3 = clean[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_counts_before[column] = int(((clean[column] < lower) | (clean[column] > upper)).sum())

    clean[column] = clean[column].clip(lower=lower, upper=upper)

print("Potential outliers before capping:")
display(pd.Series(outlier_counts_before, name="count").to_frame())


## 12. Before vs After Quality Summary


In [ ]:
quality_after = data_quality_report(clean)

summary = pd.DataFrame({
    "Metric": [
        "Row count",
        "Column count",
        "Duplicate rows",
        "Total missing cells"
    ],
    "Before": [
        len(df),
        len(df.columns),
        int(df.duplicated().sum()),
        int(df.isna().sum().sum())
    ],
    "After": [
        len(clean),
        len(clean.columns),
        int(clean.duplicated().sum()),
        int(clean.isna().sum().sum())
    ]
})

display(summary)

print("After-cleaning quality report:")
display(quality_after)


## 13. Visual Quality Checks


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df["age"].dropna(), kde=True, ax=axes[0])
axes[0].set_title("Age Before Cleaning")

sns.histplot(clean["age"].dropna(), kde=True, ax=axes[1])
axes[1].set_title("Age After Cleaning")

plt.tight_layout()
plt.show()


## Observation

The cleaned dataset retains the same passenger-level observations except for exact duplicate rows. Missing numeric values have been imputed, categorical fields have been standardized, and extreme numeric observations have been capped using documented IQR bounds.


## 14. Final Validation


In [ ]:
assert clean.isna().sum().sum() == 0, "There are still missing values."
assert clean.duplicated().sum() == 0, "Duplicate rows remain."

print("Final validation passed.")
print("Rows:", len(clean))
print("Columns:", len(clean.columns))
display(clean.head())


## 15. Export the Cleaned Dataset


In [ ]:
output_file = "cleaned_titanic.csv"
clean.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")


## Conclusion

This workflow demonstrates a reproducible data-cleaning process rather than simply deleting incomplete records.

The final dataset has documented treatment for missing values, duplicates, inconsistent formatting, outliers and data types. A before-versus-after quality report provides a transparent record of the changes made.

The cleaned CSV can now be reused for exploratory analysis or machine-learning work.
